In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# =========================
# 全局参数
# =========================
LINEWIDTH = 3.0
CURVE_N = 180
GRID_CURVE_N = 40

# 与前两个图保持一致的颜色顺序（从上到下）
ROW_COLORS = [
    "#2f6f9f",  # 第1行：蓝
    "#8d5aa7",  # 第2行：紫
    "#67a85c",  # 第3行：绿
    "#e08d32",  # 第4行：橙
    "#2b4278",  # 第5行：深蓝
]


# =========================
# 生成平滑且波动较大的序列
# =========================
def generate_smooth_series(seed=0, n=CURVE_N, trend=0.0):
    rng = np.random.default_rng(seed)
    x = np.linspace(0, 1, n)

    noise = rng.normal(0, 0.45, n)
    y = np.cumsum(noise)

    kernel = np.array([1, 2, 3, 4, 3, 2, 1], dtype=float)
    kernel /= kernel.sum()

    for _ in range(8):
        y = np.convolve(y, kernel, mode="same")

    phase1 = rng.uniform(0, 2 * np.pi)
    phase2 = rng.uniform(0, 2 * np.pi)
    y = (
        0.72 * y
        + 0.90 * np.sin(2 * np.pi * 1.5 * x + phase1)
        + 0.45 * np.sin(2 * np.pi * 3.0 * x + phase2)
        + trend * (x - 0.5) * 3.0
    )

    y = y - y.min()
    y = y / (y.max() + 1e-8)

    return x, y


# =========================
# 前两个图坐标轴样式
# =========================
def style_timeseries_axis(ax):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 5.2)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("none")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(LINEWIDTH)
    ax.spines["bottom"].set_linewidth(LINEWIDTH)
    ax.spines["left"].set_color("#3a3a3a")
    ax.spines["bottom"].set_color("#3a3a3a")


# =========================
# 图1、图2：多变量时间序列图标
# =========================
def draw_timeseries_icon(ax, seed_offset=0, variant=1):
    offsets = [4.10, 3.25, 2.45, 1.60, 0.75]

    for i, (c, offset) in enumerate(zip(ROW_COLORS, offsets)):
        trend = 0.35 if i in [0, 1] else 0.10
        x, y = generate_smooth_series(seed=seed_offset + i, trend=trend)

        scale = 0.95 if variant == 1 else 1.10
        y_plot = offset + scale * y * 0.95

        ax.plot(
            x, y_plot,
            color=c,
            linewidth=LINEWIDTH,
            solid_capstyle="round",
            solid_joinstyle="round"
        )

    style_timeseries_axis(ax)


# =========================
# 图3：网格图标 + 随机缺失块
# 每一行颜色固定，与前两个图保持一致
# =========================
def draw_patch_grid_icon_with_missing(
    ax,
    rows=5,
    cols=5,
    seed=2026,
    missing_ratio=0.20,
    cover_color_mode="white"
):
    rng = np.random.default_rng(seed)

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor("none")

    # 外框
    for spine in ax.spines.values():
        spine.set_linewidth(LINEWIDTH)
        spine.set_color("#3a3a3a")

    # 网格线
    for c in range(1, cols):
        ax.plot([c, c], [0, rows], color="#4a4a4a", linewidth=LINEWIDTH)
    for r in range(1, rows):
        ax.plot([0, cols], [r, r], color="#4a4a4a", linewidth=LINEWIDTH)

    # 原图风格里的背景高亮块
    highlight_map = {
        (0, 2): "#f0d766",
        (0, 3): "#f0d766",
        (1, 0): "#d9ea79",
        (1, 1): "#d9ea79",
        (1, 2): "#d9ea79",
        (1, 3): "#d9ea79",
        (2, 0): "#9fd392",
        (2, 1): "#d7b4dd",
    }

    # 缺失格子覆盖色
    random_cover_colors = [
        "#ffffff",
        "#f2f2f2",
        "#e6e6e6",
        "#d9d9d9",
        "#d7b4dd",
        "#f0d766",
        "#d9ea79",
        "#9fd392",
        "#cfe2f3",
    ]

    # 随机缺失格子
    total_cells = rows * cols
    missing_count = max(1, int(total_cells * missing_ratio))
    all_cells = [(r, c) for r in range(rows) for c in range(cols)]
    chosen_idx = rng.choice(len(all_cells), size=missing_count, replace=False)
    missing_cells = {all_cells[i] for i in chosen_idx}

    for rr in range(rows):
        for cc in range(cols):
            y0 = rows - 1 - rr

            # 先画背景高亮块
            if (rr, cc) in highlight_map:
                ax.add_patch(
                    Rectangle(
                        (cc, y0), 1, 1,
                        facecolor=highlight_map[(rr, cc)],
                        edgecolor="none",
                        alpha=0.95,
                        zorder=1
                    )
                )

            # 如果该格子缺失，则直接纯色覆盖
            if (rr, cc) in missing_cells:
                if cover_color_mode == "white":
                    cover_color = "#ffffff"
                elif cover_color_mode == "gray":
                    cover_color = "#e6e6e6"
                else:
                    cover_color = rng.choice(random_cover_colors)

                ax.add_patch(
                    Rectangle(
                        (cc, y0), 1, 1,
                        facecolor=cover_color,
                        edgecolor="none",
                        alpha=1.0,
                        zorder=2
                    )
                )
                continue

            # 非缺失格子：画小曲线
            # 关键修改：同一行的颜色固定
            curve_color = ROW_COLORS[rr]

            xs = np.linspace(cc + 0.10, cc + 0.90, GRID_CURVE_N)

            noise = rng.normal(0, 0.50, GRID_CURVE_N)
            ys = np.cumsum(noise)

            kernel = np.array([1, 2, 3, 4, 3, 2, 1], dtype=float)
            kernel /= kernel.sum()

            for _ in range(5):
                ys = np.convolve(ys, kernel, mode="same")

            ys = ys - ys.min()
            ys = ys / (ys.max() + 1e-8)
            ys = y0 + 0.15 + 0.70 * ys

            ax.plot(
                xs, ys,
                color=curve_color,
                linewidth=LINEWIDTH,
                solid_capstyle="round",
                solid_joinstyle="round",
                zorder=3
            )


# =========================
# 保存三张 SVG
# =========================
def save_three_icons():
    # 图1
    fig1, ax1 = plt.subplots(figsize=(3.2, 2.8))
    draw_timeseries_icon(ax1, seed_offset=10, variant=1)
    fig1.savefig(
        "icon_ts_1.svg",
        format="svg",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig1)

    # 图2
    fig2, ax2 = plt.subplots(figsize=(3.2, 2.8))
    draw_timeseries_icon(ax2, seed_offset=50, variant=2)
    fig2.savefig(
        "icon_ts_2.svg",
        format="svg",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig2)

    # 图3
    fig3, ax3 = plt.subplots(figsize=(3.2, 3.2))
    draw_patch_grid_icon_with_missing(
        ax3,
        rows=5,
        cols=5,
        seed=2026,
        missing_ratio=0.20,      # 缺失比例，可自行调整
        cover_color_mode="white" # 可选: "white" / "gray" / "random"
    )
    fig3.savefig(
        "icon_grid_missing.svg",
        format="svg",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )
    plt.close(fig3)

    print("已生成三张 SVG 图片：")
    print("1) icon_ts_1.svg")
    print("2) icon_ts_2.svg")
    print("3) icon_grid_missing.svg")


if __name__ == "__main__":
    save_three_icons()

已生成三张 SVG 图片：
1) icon_ts_1.svg
2) icon_ts_2.svg
3) icon_grid_missing.svg


In [4]:
import numpy as np
import matplotlib.pyplot as plt


LINEWIDTH = 2.0


def gaussian(x, mu, sigma, height):
    """
    非标准化高斯形状：
    mu 控制中心位置
    sigma 控制宽度
    height 控制峰值高度
    """
    return height * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def draw_distribution_icon(output_file="distribution_icon.svg"):
    x = np.linspace(0, 10, 1200)

    # 每个分布：mu, sigma, height, 边框色, 填充色
    # 注意：这里的 sigma 和 height 都不一样
    distributions = [
        (3.10, 0.78, 1.18, "#4b2e83", "#cbb8e8"),  # 紫色：更高、更窄
        (4.20, 0.72, 1.05, "#5d8ec2", "#cfe0f5"),  # 蓝色1：较高、较窄
        (5.10, 0.82, 0.88, "#4f78a8", "#d9e7f7"),  # 蓝色2：中等高度
        (6.50, 1.05, 0.78, "#d08a42", "#efd6b8"),  # 橙色：更宽、更矮
    ]

    fig, ax = plt.subplots(figsize=(3.8, 2.2))

    for mu, sigma, height, edge_color, fill_color in distributions:
        y = gaussian(x, mu, sigma, height)

        ax.fill_between(
            x,
            y,
            0,
            color=fill_color,
            alpha=0.45,
            zorder=1
        )

        ax.plot(
            x,
            y,
            color=edge_color,
            linewidth=LINEWIDTH,
            solid_capstyle="round",
            solid_joinstyle="round",
            zorder=2
        )

    # 底部横线
    ax.plot(
        [0.55, 9.45],
        [0, 0],
        color="#666666",
        linewidth=LINEWIDTH,
        solid_capstyle="round",
        zorder=3
    )

    # 去掉坐标轴和背景
    ax.set_xlim(0.55, 9.45)
    ax.set_ylim(-0.03, 1.28)
    ax.axis("off")
    ax.set_facecolor("none")

    plt.savefig(
        output_file,
        format="svg",
        transparent=True,
        bbox_inches="tight",
        pad_inches=0.02
    )

    plt.close(fig)
    print(f"SVG saved to: {output_file}")


if __name__ == "__main__":
    draw_distribution_icon("distribution_icon.svg")

SVG saved to: distribution_icon.svg
